In [ ]:
%pip install torch --index-url https://download.pytorch.org/whl/cu128
%pip install "numpy>=1.26,<2" "pandas>=2.0,<3" "protobuf>=4.25,<5" "transformers>=5,<6" "huggingface-hub>=1.3,<2" "datasets>=4,<5.1" "pyarrow>=21,<24" safetensors sentencepiece

import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

Looking in indexes: https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\happywh(user)\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   - -------------------------------------- 0.8/27.5 MB 4.8 MB/s eta 0:00:06
   --- ------------------------------------ 2.4/27.5 MB 6.1 MB/s eta 0:00:05
   ----- ---------------------------------- 3.9/27.5 MB 6.3 MB/s eta 0:00:04
   ------- -------------------------------- 5.0/27.5 MB 6.2 MB/s eta 0:00:04
   --------- ------------------------------ 6.3/27.5 MB 6.1 MB/s eta 0:00:04
   ----------- ---------------------------- 7.6/27.5 MB 6.2 MB/s eta 0:00:04
   ------------ --------------------------- 8.7/27.5 MB 6.2 MB/s eta 0:00:04
   -------------- ------------------------- 10.0/27.5 MB 6.0 MB/s eta 0:00:03
   --------------- ------------------------ 10.7/27.5 MB 5.9 MB/s eta 0:00:03
   ---------------- ----------------------- 11.5/27.5 MB 5.6 MB/s eta 0:00:03
   ----------------- ---------------------- 12.1/27.5 MB 5.4 MB/s eta 0:00:03
   -----


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\happywh(user)\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


{'status': 'ok', 'restart': True}

In [2]:
import torch
print(torch.__version__)                 # should show +cu128 or newer
print(torch.cuda.is_available())         # True
#print(torch.cuda.get_device_name(0)) 
#print(torch.cuda.get_device_capability(0))

2.14.1+cpu
False


In [3]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from pathlib import Path
import pandas as pd
import random, os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import AutoModel
import torch.nn.functional as F

DATA_DIR = Path("./data")
print("working dir:", Path.cwd())
print("data dir   :", DATA_DIR.resolve())
print("files      :", [p.name for p in DATA_DIR.iterdir()])

class CFG:
    seed = 42
    max_length = 128
    lr_encoder = 2e-5
    lr_heads = 1e-3
    train_batch_size = 16
    eval_batch_size = 64
    train_epochs = 3
    patience = 8
    weight_decay = 0.01
    warmup_ratio = 0.1
    label_smoothing = 0.1
    reason_loss_weight = 0.2 # weight for reason loss in the total overall loss. Ratio to the weight of slot loss 
    strength_loss_weight = 1.0 # weight for strength loss in the total overall loss. Ratio to the weight of slot loss
    early_stopping = False #False for normal Epoch stopping, True to use patience and valuation score to stop the training
    model_name = "microsoft/deberta-v3-base" 
    custom_name = None #for custom_layers, look under initialise model, custom layers for possible inputs
    custom_layers_pool_conc = 1 #number of layers for pooling or concetanation

#########################
# Models experimented with (to put any one under model_name for testing)
# "microsoft/deberta-v3-base"
# "bert-base-uncased"
# "roberta-base"
# "distilbert-base-uncased"
########################

# snapshot of the class defaults, taken once, right after the class
DEFAULTS = {k: v for k, v in vars(CFG).items() if not k.startswith("__")}
    
def seed_everything(seed):
    import random, os
    import numpy as np
    import torch
    
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

working dir: D:\uni stuff\CS425-Generative AI for Natural Language Processing\testing\model_training
data dir   : D:\uni stuff\CS425-Generative AI for Natural Language Processing\testing\model_training\data
files      : ['extractor.jsonl']


In [4]:
###################################################################
# Load Tokenizer, Model and Sample Data
###################################################################
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
df = pd.read_json(DATA_DIR / "extractor.jsonl", lines=True)

###################################################################
# 2. Tokenize and Split Dataset 
###################################################################
# ---- Encode RIASEC slots and reason  ----
TYPES   = ["R","I","A","S","E","C"]
POL     = {"negative": 0, "neutral": 1, "positive": 2} #positions are defined for cross entropy later
REASON  = {"enjoyment": 0, "familiarity": 1, "neither": 2}

df["context"]  = df["context"].fillna("") #some rows do not have context, just basic answer, e.g. "I like playing sports"

def slots_to_numbers(slots):
    numbers = []
    for t in TYPES:                    
        if slots is not None and t in slots:
            word = slots[t]              # it will either be positive/positive
        else:
            word = "neutral"             # if there is no type, the statement is neutral
        numbers.append(POL[word])
    return numbers

y_slots = []
y_reason = []
for i in range(len(df)):
    y_slots.append(slots_to_numbers(df["slots"].iloc[i])) # encodes negative neutral and positive into numbers
    y_reason.append(REASON[df["reason"].iloc[i]])   # encodes enjoyment familiarity and neither into numbers

#add to df to access later
df["y_slots"] = y_slots 
df["y_reason"] = y_reason

#------Encode strength -------------------------------
STR = {"dontmind": 1, "mild": 2, "strong": 3} # dontmind = 0.2, mild = 0.55, strong = 1
STR_NAMES = ["inactive", "dontmind", "mild", "strong"]

def strengths_to_classes(labels, y_slots):
    out = []
    for i in range(6):
        if y_slots[i] == 1:
            out.append(0)                          # inactive: if the slot is neutral, score = 0
        else:
            out.append(STR[labels[TYPES[i]]])      # errors here if an active type has no strength label
    return out

all_rows = []
for i in range(len(df)):                                   # one pass per sentence
    labels  = df["strength_label"].iloc[i]                 # e.g. {"R": "mild", "I": "mild"}
    y_slots = df["y_slots"].iloc[i]                        # e.g. [2, 2, 1, 1, 1, 1]
    all_rows.append(strengths_to_classes(labels, y_slots)) # returns a list of 6 numbers

df["y_strength"] = all_rows                                # one list of 6 per row

class ExtractorDataset(Dataset):
    def __init__(self, frame):
        self.items = []
        for r in frame.itertuples():
            if r.context:
                enc = tokenizer(r.context, r.text, truncation="only_first", 
                                max_length=CFG.max_length, return_token_type_ids=False) #tokenises both context and text, used for rows with context
            else:
                enc = tokenizer(r.text, truncation=True, 
                                max_length=CFG.max_length, return_token_type_ids=False) #tokenises just text, used for statements that do not need context
            self.items.append(dict(ids=enc["input_ids"], slots=r.y_slots, reason=r.y_reason,
                       strength=r.y_strength, w=float(r.weight))) 
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def collate(batch):
    padded = tokenizer.pad([{"input_ids": b["ids"]} for b in batch], return_tensors="pt") #pad the inputs as each sentence can be of different length
    return dict(
        input_ids=padded["input_ids"],
        attention_mask=padded["attention_mask"], #indicates which positions are actual words and which ones are padding
        slots=torch.tensor([b["slots"] for b in batch]),
        reason=torch.tensor([b["reason"] for b in batch]),
        w=torch.tensor([b["w"] for b in batch], dtype=torch.float),
        strength=torch.tensor([b["strength"] for b in batch]),
    )

ds = {s: ExtractorDataset(df[df["split"] == s]) for s in ("train", "val", "test")}

g = torch.Generator().manual_seed(CFG.seed)          # repeatable batch order
train_dl = DataLoader(ds["train"], batch_size=CFG.train_batch_size, shuffle=True,
                      collate_fn=collate, generator=g)
val_dl   = DataLoader(ds["val"],  batch_size=CFG.eval_batch_size, shuffle=False, collate_fn=collate)
test_dl  = DataLoader(ds["test"], batch_size=CFG.eval_batch_size, shuffle=False, collate_fn=collate)


In [5]:
###################################################################
# 3. Initialise Model
###################################################################
seed_everything(CFG.seed)      # set a seed before creating the model, for repeatability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Is using Cuda?: ", torch.cuda.is_available())
use_bf16 = device.type == "cuda" and torch.cuda.is_bf16_supported() #helps with training stability by containing more bits

# 18 slot outputs (6 RIASEC types x 3 polarities) + 3 reason outputs + 6 strength score x 4 possible categories
if CFG.custom_name == None:   
    model = AutoModelForSequenceClassification.from_pretrained(CFG.model_name, num_labels=45).to(device) #for just plain pre-trained models
    

###################################################################
# Custom layers. Put any class name intp custom_name to run it
###################################################################

class CustomLayers1(nn.Module): #adding custom layers to the pre-trained models

    def __init__(self, model_name):
        super().__init__()
        
        # Pretrained encoder according to model
        self.encoder = AutoModel.from_pretrained(model_name)
        self.config = self.encoder.config
        hidden_size = self.encoder.config.hidden_size

        # Custom classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(512, 45)
        )

    def forward(self, input_ids, attention_mask):

        # Pass input through pretrained encoder
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Extract first token representation
        features = outputs.last_hidden_state[:, 0, :]

        # Pass representation through custom head
        logits = self.classifier(features)

        return logits

class CustomLayers2(nn.Module): #adding custom layers to the pre-trained models

    def __init__(self, model_name):
        super().__init__()
        
        # Pretrained encoder according to model
        self.encoder = AutoModel.from_pretrained(model_name)
        self.config = self.encoder.config
        hidden_size = self.encoder.config.hidden_size

        # Custom classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 45)
        )

    def forward(self, input_ids, attention_mask):

        # Pass input through pretrained encoder
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Extract first token representation
        features = outputs.last_hidden_state[:, 0, :]

        # Pass representation through custom head
        logits = self.classifier(features)

        return logits

class CustomLayers3(nn.Module): #adding custom layers to the pre-trained models

    def __init__(self, model_name):
        super().__init__()
        
        # Pretrained encoder according to model
        self.encoder = AutoModel.from_pretrained(model_name)
        self.config = self.encoder.config
        hidden_size = self.encoder.config.hidden_size

        # Custom classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 45)
        )

    def forward(self, input_ids, attention_mask):

        # Pass input through pretrained encoder
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Extract first token representation
        features = outputs.last_hidden_state[:, 0, :]

        # Pass representation through custom head
        logits = self.classifier(features)

        return logits
        
class CustomPooling(nn.Module):
    def __init__(self, model_name, dropout=0.1):
        super().__init__() 
        try: # some models have pooling layer (roberta) others dun have (deberta) 
            self.encoder = AutoModel.from_pretrained(model_name, add_pooling_layer=False)
        except TypeError:
            self.encoder = AutoModel.from_pretrained(model_name)
        self.config = self.encoder.config
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, 45)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )

        # take the last n_layers for pooling
        hs = outputs.hidden_states[-CFG.custom_layers_pool_conc:]
        hidden = torch.stack(hs, dim=0).mean(dim=0)               # (B, T, H)

        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)      # (B, T, 1)
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)  # (B, H)

        return self.classifier(self.dropout(pooled))

class CustomConcat(nn.Module):
    def __init__(self, model_name, dropout=0.1):
        super().__init__() 
        try: # some models have pooling layer (roberta) others dun have (deberta) 
            self.encoder = AutoModel.from_pretrained(model_name, add_pooling_layer=False)
        except TypeError:
            self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.config = self.encoder.config

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * CFG.custom_layers_pool_conc, 45) #intput to the classifier is larger cause of concetanation 

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )

        # concatenate the last n_layers for pooling
        hs = outputs.hidden_states[-CFG.custom_layers_pool_conc:]
        mask = attention_mask.unsqueeze(-1).to(hs[0].dtype)       # (B, T, 1)
        denom = mask.sum(1).clamp(min=1e-9)                       # (B, 1)

        pooled = torch.cat([(h * mask).sum(1) / denom for h in hs], dim=-1)  # (B, n * H)

        return self.classifier(self.dropout(pooled))

class CustomConcatWithLayer(nn.Module):
    def __init__(self, model_name, dropout=0.1):
        super().__init__()
        try:  # some models have a pooling layer (roberta), others don't (deberta)
            self.encoder = AutoModel.from_pretrained(model_name, add_pooling_layer=False)
        except TypeError:
            self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.config = self.encoder.config
        
        self.dropout = nn.Dropout(dropout)
        # input to the classifier is larger because of concatenation
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * CFG.custom_layers_pool_conc, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 45),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )

        # concatenate the last n_layers for pooling
        hs = outputs.hidden_states[-CFG.custom_layers_pool_conc:]
        mask = attention_mask.unsqueeze(-1).to(hs[0].dtype)       # (B, T, 1)
        denom = mask.sum(1).clamp(min=1e-9)                       # (B, 1)

        pooled = torch.cat([(h * mask).sum(1) / denom for h in hs], dim=-1)  # (B, n * H)

        return self.classifier(self.dropout(pooled))

if CFG.custom_name:
    model_class = globals()[CFG.custom_name]
    # 2. Instantiate and move to the device
    model = model_class(CFG.model_name).to(device)

def forward_heads(model, input_ids, attention_mask):
    if CFG.custom_name:
        logits = model(input_ids=input_ids, attention_mask=attention_mask).float()
    else:
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits.float()
    return (
        logits[:, :18].view(-1, 6, 3),
        logits[:, 18:21],
        logits[:, 21:45].view(-1, 6, 4)
    )

# slightly boosts the weights of RIASEC slots and Reasons that do not appear as much in the training data
def sqrt_inv(counts):
    w = 1.0 / np.sqrt(np.maximum(counts, 1))
    return torch.tensor(w / w.mean(), dtype=torch.float, device=device)

tr = df[df["split"] == "train"]
slot_counts   = np.bincount(np.concatenate(tr["y_slots"].tolist()), minlength=3) #counts how many 0s, 1s and 2s
reason_counts = np.bincount(tr["y_reason"], minlength=3) #counts how many 0s, 1s and 2s
str_counts = np.bincount(np.concatenate(tr["y_strength"].tolist()), minlength=4) # counts how many 0, 1s, 2s, and 3s

slot_cw, reason_cw, str_cw  = sqrt_inv(slot_counts), sqrt_inv(reason_counts), sqrt_inv(str_counts)
print("slot weights:", slot_cw.tolist(), "| reason weights:", reason_cw.tolist(), "| strength counts:", str_counts, "weights:", str_cw.tolist())

Is using Cuda?:  False


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den

slot weights: [1.4606754779815674, 0.41670846939086914, 1.1226160526275635] | reason weights: [0.41126343607902527, 1.204404354095459, 1.3843321800231934] | strength counts: [7323   68  998  539] weights: [0.2250705361366272, 2.3356549739837646, 0.609674334526062, 0.8296000957489014]


In [ ]:
###################################################################
# 4. Training and evaluation
###################################################################
from pathlib import Path
from transformers import get_linear_schedule_with_warmup

OUT_DIR = Path("./riasec_model")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_PATH = OUT_DIR / "best_model.pt" 

#calculate f1 score with numpy (the f1 score from sklearn faced a dll loading issue) 
def f1_score(y_true, y_pred, labels):
    scores = []
    for c in labels:
        tp = np.sum((y_true == c) & (y_pred == c))
        fp = np.sum((y_true != c) & (y_pred == c))
        fn = np.sum((y_true == c) & (y_pred != c))
        d = 2 * tp + fp + fn
        scores.append(2 * tp / d if d > 0 else 0.0)
    return float(np.mean(scores))

# Loss (same function for train and validation)
def compute_loss(slot_logits, reason_logits, strength_logits, batch, smooth=0.0):
    ce_slot = F.cross_entropy(
        slot_logits.reshape(-1, 3), batch["slots"].reshape(-1), #flattens the output
        weight=slot_cw, reduction="none", label_smoothing=smooth, # weight multiplies the loss with the slot weights
    ).view(-1, 6).mean(1) # Assigns each of the R,I,A,S,E,C slot a loss, then takes the mean for overall sample score 
 
    ce_reason = F.cross_entropy(
        reason_logits, batch["reason"],
        weight=reason_cw, reduction="none", label_smoothing=smooth,) #for each sample, assign a weighted loss 
    
    ce_strength = F.cross_entropy(
        strength_logits.reshape(-1, 4), batch["strength"].reshape(-1),
        weight=str_cw, reduction="none", label_smoothing=smooth,
    ).view(-1, 6).mean(1)      
 
    per_example = ce_slot + CFG.reason_loss_weight * ce_reason + CFG.strength_loss_weight * ce_strength
    return (per_example * batch["w"]).sum() / batch["w"].sum() #takes the weight of each sentence and gets weighted average for overall loss in the batch
    

@torch.no_grad() #disables gradient calculation, less memory usage
#generating the raw, unnormalized score outputs from the final layer of the model (used as part of model evaluation), also calculates avg loss of batches
def predict_logits(dl):
    model.eval()
    keys = ["slot", "reason", "strength", "y_slot", "y_reason", "y_strength"]
    out = {k: [] for k in keys}
    total, n = 0.0, 0
    for b in dl:
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast(device.type, dtype=torch.bfloat16, enabled=use_bf16):
            s, r, t = forward_heads(model, b["input_ids"], b["attention_mask"])
        total += compute_loss(s.float(), r.float(), t.float(), b).item()
        n += 1
        for k, v in zip(keys, [s, r, t, b["slots"], b["reason"], b["strength"]]):
            out[k].append(v.cpu())
    out = {k: torch.cat(v) for k, v in out.items()}
    out["val_loss"] = total / n   # mean of per-batch weighted losses
    return out

#changes the logits into scores
def evaluate(out):
    ps, pr = out["slot"].argmax(-1).numpy(), out["reason"].argmax(-1).numpy() #chooses the output with the highest score
    ys, yr = out["y_slot"].numpy(), out["y_reason"].numpy()
    
    #scoring the slots head
    per_type = {}
    for i in range(6):                       # i = 0 is R, 1 is I, 2 is A, 3 is S, 4 is E, 5 is C
        true_labels = ys[:, i]               # the true answers for this one type
        predicted   = ps[:, i]               # the model's answers for this one type
        per_type[TYPES[i]] = f1_score(true_labels, predicted, [0, 1, 2]) #calculate f1 score
    scores = list(per_type.values())         
    slot_f1 = sum(scores) / len(scores)      # average of the six scores

    #scoring the reason head
    reason_f1 = f1_score(yr, pr, [0, 1, 2]) #scores the reason head on all rows.
    active = (ys != 1).any(1) #filters the non-neutral, i.e. relevant statements
    reason_f1_active = f1_score(yr[active], pr[active], [0, 1]) #only evaluate the scores of the relevant sentences

    #scoring the strength head
    slot_true       = out["y_slot"].numpy()                  # [N, 6]  true polarity (1 = neutral)
    strength_true   = out["y_strength"].numpy()              # [N, 6]  true strength class (0..3)
    strength_scores = out["strength"][:, :, 1:].numpy()      # [N, 6, 3]  scores for dontmind, mild, strong only
    
    true_list, pred_list = [], []
    for n in range(len(slot_true)):               
        for i in range(6):                        
            if slot_true[n, i] != 1:              # only types that are active gets scored
                true_list.append(strength_true[n, i])
                pred_list.append(strength_scores[n, i].argmax() + 1)   # 0..2 becomes 1..3
    
    strength_f1 = f1_score(np.array(true_list), np.array(pred_list), [1, 2, 3])
    
    return dict(slot_f1=slot_f1, reason_f1=reason_f1, reason_f1_active=reason_f1_active,
                strength_f1=strength_f1, per_type=per_type,
                score=(slot_f1 + reason_f1_active + strength_f1) / 3)
    
    
# optimiser: small lr for the encoder, larger for the new head
no_decay = ("bias", "LayerNorm.weight") #parameters that should not be pulled toward zero by weight decay.

if CFG.custom_name: #custom layers initialisation here, the encoder and parameters are all different names for diff pre-trained models
    enc = list(model.encoder.named_parameters())
    head_params = list(model.classifier.parameters())
elif CFG.model_name == "microsoft/deberta-v3-base":
    #Deberta config
    enc = list(model.deberta.named_parameters()) #pretrained encoder
    head_params = list(model.pooler.parameters()) + list(model.classifier.parameters()) #randomly initialised parameters
elif CFG.model_name == "bert-base-uncased":
    #Bert config
    enc = list(model.bert.named_parameters())
    head_params = list(model.classifier.parameters())
elif CFG.model_name == "roberta-base":
    #RoBERTa config
    enc = list(model.roberta.named_parameters()) 
    head_params = list(model.classifier.parameters())
elif CFG.model_name == "distilbert-base-uncased":
    #DistillBert config
    enc = list(model.distilbert.named_parameters())
    head_params = list(model.pre_classifier.parameters()) + list(model.classifier.parameters())

optimizer = torch.optim.AdamW([
    {"params": [p for n, p in enc if not any(k in n for k in no_decay)], "lr": CFG.lr_encoder, "weight_decay": CFG.weight_decay}, #encoder weights
    {"params": [p for n, p in enc if any(k in n for k in no_decay)],     "lr": CFG.lr_encoder, "weight_decay": 0.0}, #encoder biases and layer norms
    {"params": head_params, "lr": CFG.lr_heads, "weight_decay": 0.0}, #New heads for training, larger head learning rate
])

total_steps = CFG.train_epochs * len(train_dl)

#The learning rate rises linearly from 0 to its peak over the first warmup_ratio * total_steps optimizer steps, then decays linearly to 0.
# This avoids large early updates (from the randomly initialised head) wrecking pretrained encoder weights.
scheduler = get_linear_schedule_with_warmup(optimizer, int(CFG.warmup_ratio * total_steps), total_steps) 

# training loop
best = dict(score=-1.0, val_loss=1e9, epoch=0)
bad = 0
     
for epoch in range(1, CFG.train_epochs + 1):
    model.train()
    running = 0.0
 
    for b in train_dl:
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast(device.type, dtype=torch.bfloat16, enabled=use_bf16):
            s, r, t = forward_heads(model, b["input_ids"], b["attention_mask"])
        loss = compute_loss(s.float(), r.float(), t.float(), b, smooth=CFG.label_smoothing)
 
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running += loss.item()

    #scoring the current model using the validation set
    out = predict_logits(val_dl)
    m = evaluate(out)
    val_loss = out["val_loss"]
 
    tie = abs(m["score"] - best["score"]) <= 1e-4 #1e-4 is the min delta to pass so as to reduce the impact of training noise when deciding the best score
    improved = m["score"] > best["score"] + 1e-4 or (tie and val_loss < best["val_loss"])
 
    print(f"epoch {epoch:2d} | train loss {running/len(train_dl):.3f} | val loss {val_loss:.3f} | "
          f"slot F1 {m['slot_f1']:.3f} | reason F1 {m['reason_f1']:.3f} | "
          f"active rows {m['reason_f1_active']:.3f} | strength F1 {m['strength_f1']:.3f} | "
          f"score {m['score']:.3f} {'<- best' if improved else ''}")

    if CFG.early_stopping: #early stopping based on best score and patience
        if improved:
            bad = 0
            best = dict(score=m["score"], val_loss=val_loss, epoch=epoch)
            torch.save({"state": model.state_dict(), "epoch": epoch,
                        "score": m["score"], "seed": CFG.seed}, BEST_PATH)
        else:
            bad += 1
            if bad >= CFG.patience:
                print("early stopping")
                break

if BEST_PATH.exists() and CFG.early_stopping: #if stop early, load the best model as the main model based on the score. else, load the latest model
    ckpt = torch.load(BEST_PATH, map_location="cpu")
    model.load_state_dict(ckpt["state"])
    ckpt = torch.load(BEST_PATH, map_location="cpu")
    model.load_state_dict(ckpt["state"])
    print("restored epoch", ckpt["epoch"], "| val score", round(ckpt["score"], 4))


In [ ]:
###################################################################
# 5. Saving the model
###################################################################
from safetensors.torch import save_model
OUT_DIR.mkdir(exist_ok=True)

names = [f"{t}:{p}" for t in TYPES for p in ("negative", "neutral", "positive")] \
        + ["enjoyment", "familiarity", "neither"] \
        + [f"{t}:strength:{s}" for t in TYPES for s in STR_NAMES]
model.config.id2label = {i: n for i, n in enumerate(names)}
model.config.label2id = {n: i for i, n in enumerate(names)}

model.config.extractor = dict(              # extra settings are inside config.json
    types=TYPES, max_len=CFG.max_length, seed=CFG.seed,
    best_epoch=best["epoch"], best_val_score=best["score"],
    T_slot=1.0, T_reason=1.0, T_strength=1.0, #T_slot and T_reason are used to adjust the temperatures later
    custom_layers_pool_conc=CFG.custom_layers_pool_conc, custom_name=CFG.custom_name,
    model_name=CFG.model_name,
)

if CFG.custom_name:
    save_model(model, str(OUT_DIR / "model.safetensors"))
    # config.json with id2label, label2id and extractor settings
    model.config.save_pretrained(OUT_DIR)
else:
    model.save_pretrained(OUT_DIR)  # save config.json and model.safetensors
tokenizer.save_pretrained(OUT_DIR)
print(sorted(p.name for p in OUT_DIR.iterdir()))

In [ ]:
###################################################################
# 5. Loading the model
###################################################################
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from safetensors.torch import load_model
from transformers import AutoConfig

OUT_DIR = Path("./riasec_model")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_bf16 = device.type == "cuda" and torch.cuda.is_bf16_supported()

# ---- load the saved model ----
saved_cfg = AutoConfig.from_pretrained(OUT_DIR)  # reads saved config.json
cfg = saved_cfg.extractor # seed, max_len, T_slot, T_reason, T_strength, best epoch

tok = AutoTokenizer.from_pretrained(OUT_DIR)

if CFG.custom_name:  # custom model
    print(CFG.custom_name)
    model_class = globals()[CFG.custom_name]
    # 2. Instantiate and move to the device
    model = model_class(CFG.model_name).to(device)    
    load_model(model, str(OUT_DIR / "model.safetensors"))
    model.config = saved_cfg                
else: # normal pre-trained model
    model = AutoModelForSequenceClassification.from_pretrained(OUT_DIR) #for models that do not have additional layers

model = model.to(device).eval()

#helpers for test report and predict()
def forward_heads(model, input_ids, attention_mask):
    if CFG.custom_name:
        logits = model(input_ids=input_ids, attention_mask=attention_mask).float()
    else:
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits.float() #this is for huggingface models without add layers
    return (logits[:, :18].view(-1, 6, 3),        # slots
            logits[:, 18:21],                     # reason
            logits[:, 21:45].view(-1, 6, 4))      # strength: inactive, dontmind, mild, strong

@torch.no_grad()
def predict_logits(dl):
    keys = ["slot", "reason", "strength", "y_slot", "y_reason", "y_strength"]
    out = {k: [] for k in keys}
    for b in dl:
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast(device.type, dtype=torch.bfloat16, enabled=use_bf16):
            s, r, t = forward_heads(model, b["input_ids"], b["attention_mask"])
        for k, v in zip(keys, [s, r, t, b["slots"], b["reason"], b["strength"]]):
            out[k].append(v.cpu())
    return {k: torch.cat(v) for k, v in out.items()}

In [ ]:
pols    = ["negative", "neutral", "positive"]
reasons = ["enjoyment", "familiarity", "neither"]
types   = ["Realistic", "Investigative", "Artistic", "Social", "Enterprising", "Conventional"]
strength_scores = [0.2, 0.55, 1] # dontmind = 0.2, mild = 0.55, strong = 1
str_names = ["dontmind", "mild", "strong"]

levels = torch.tensor(strength_scores)

@torch.no_grad()
def predict(text, context=None):
    args = (context, text) if context else (text,)
    enc = tok(*args, truncation="only_first" if context else True, max_length=cfg["max_len"],
              return_tensors="pt", return_token_type_ids=False).to(device)
    out = model(**enc)
    if CFG.custom_name:   # custom model
        lg = model(**enc)[0].float().cpu()            # 45 scores
    else:                 # pretrained model 
        lg = model(**enc).logits[0].float().cpu()     # 45 scores

    sp  = torch.softmax(lg[:18].view(6, 3), -1)              # [6, 3]  polarity per type
    rp  = torch.softmax(lg[18:21], -1)                       # [3]     reason
    # strength: 6 types x 4 classes; drop the "inactive" column, keep dontmind / mild / strong
    stp = torch.softmax(lg[21:45].view(6, 4)[:, 1:], -1)   # [6, 3]

    slots, strength = {}, {}
    for i in range(6):
        pol = pols[int(sp[i].argmax())]
        slots[types[i]] = pol
        if pol != "neutral":          # strength only means something for active types
            k = int(stp[i].argmax())  # 0..2, same index for the name and the score
            strength[types[i]] = dict(
                label=str_names[k],
                conf=round(float(stp[i].max()), 3),
                score=round(float((stp[i] * levels).sum()), 3), # smooth 0..1 value
            )
    j = int(rp.argmax())

    slot_confidences = [float(sp[i].max()) for i in range(6)]
    reason_confidence = float(rp.max())
    overall_confidence = float((sum(slot_confidences) / len(slot_confidences) + reason_confidence) / 2)

    return dict(slots=slots, reason=(reasons[j], round(float(rp[j]), 3)), seed=cfg["seed"],
                strength=strength, confidence=round(overall_confidence, 3))

def show(text, context=None):
    r = predict(text, context)

    active = {}
    for slot, label in r["slots"].items():
        if label != "neutral":
            st = r["strength"][slot]
            active[slot] = f"{label}, {st['label']} (score {st['score']:.2f}, conf {st['conf']:.2f})"

    output = text, "\n  ", active or "all neutral", r["reason"], f"confidence={r['confidence']:.3f}", "seed", r["seed"]
    return output


test_sentences = [
    # Neutral
    "I'm not sure what to pick for next term.",
    "Can you tell me what you need from me?",
    "My classes end at three on Wednesdays.",
    "The hospital is a few stops from my house.",
    "I had rice for lunch.",
    "Honestly, I could take it or leave it.",
    "The stadium was empty on Monday.",
    "I'm waiting to hear back from the school.",
    "It snowed a little in the north last week.",
    "I'm twenty and live with my grandparents.",
    "The bakery on the corner opens at six.",
    "I don't know yet, I'll think about it.",
    "sun and flowers, clouds and nice things",
    
    # Realistic (R)
    "I strongly enjoy repairing engines.",
    "I enjoy repairing engines.",
    "I somewhat enjoy repairing engines.",
    "I have no particular interest in repairing engines.",
    "I strongly dislike repairing engines.",

    # Investigative (I)
    "I strongly enjoy investigating scientific problems.",
    "I enjoy investigating scientific problems.",
    "I somewhat enjoy investigating scientific problems.",
    "I have no particular interest in investigating scientific problems.",
    "I strongly dislike investigating scientific problems.",

    # Artistic (A)
    "I strongly enjoy creating original artwork.",
    "I enjoy creating original artwork.",
    "I somewhat enjoy creating original artwork.",
    "I have no particular interest in creating original artwork.",
    "I strongly dislike creating original artwork.",

    # Social (S)
    "I strongly enjoy helping people solve personal problems.",
    "I enjoy helping people solve personal problems.",
    "I somewhat enjoy helping people solve personal problems.",
    "I have no particular interest in helping people solve personal problems.",
    "I strongly dislike helping people solve personal problems.",

    # Enterprising (E)
    "I strongly enjoy leading teams and persuading people.",
    "I enjoy leading teams and persuading people.",
    "I somewhat enjoy leading teams and persuading people.",
    "I have no particular interest in leading teams and persuading people.",
    "I strongly dislike leading teams and persuading people.",

    # Conventional (C)
    "I strongly enjoy organizing records and following procedures.",
    "I enjoy organizing records and following procedures.",
    "I somewhat enjoy organizing records and following procedures.",
    "I have no particular interest in organizing records and following procedures.",
    "I strongly dislike organizing records and following procedures.",
]

for sentence in test_sentences: 
    output = show(sentence) 
    print(output)

In [ ]:
###################################################################
# Test report
###################################################################
out = predict_logits(test_dl)                                  # test_dl has shuffle=False, so rows line up with df

slot_true   = out["y_slot"].numpy()                            # [N, 6]  0 negative, 1 neutral, 2 positive
slot_pred   = out["slot"].argmax(-1).numpy()
reason_true = out["y_reason"].numpy()                          # [N]
reason_pred = out["reason"].argmax(-1).numpy()
str_true    = out["y_strength"].numpy()                        # [N, 6]  0 inactive, 1 dontmind, 2 mild, 3 strong
str_pred    = out["strength"][..., 1:].argmax(-1).numpy() + 1  # ignore the inactive output; classes 1..3

def pct(x):
    return round(100 * float(x), 1)

N = len(slot_true)
active      = slot_true != 1                                   # [N, 6] type is truly active in this sentence
active_rows = active.any(1)                                    # sentence has at least one active type

print(f"Test sentences: {N} | active (sentence, type) pairs: {int(active.sum())} | rows with an active type: {int(active_rows.sum())}\n")

# ---------------- RIASEC ----------------
print("RIASEC (polarity: negative / neutral / positive)")
print(f"  active types labelled correctly:        {pct((slot_pred == slot_true)[active].mean())}%")
print("  per type (active pairs only):")
for i in range(6):
    a = active[:, i]
    n = int(a.sum())
    right = (slot_pred[:, i] == slot_true[:, i])[a].mean() if n > 0 else float("nan")
    print(f"    {TYPES[i]:<14} {pct(right):>5}%   (n = {n})")

# ---------------- reason ----------------
print("\nReason (enjoyment / familiarity / neither)")
print(f"  rows with an active type:               {pct((reason_pred == reason_true)[active_rows].mean())}%")
for c, name in enumerate(["enjoyment", "familiarity", "neither"]):
    rows = reason_true == c
    if rows.sum() > 0:
        print(f"    true {name:<12} {pct((reason_pred == reason_true)[rows].mean()):>5}%   (n = {int(rows.sum())})")
print(f"  baseline, always 'enjoyment':           {pct((reason_true == 0).mean())}%")

# ---------------- strength ----------------
print("\nStrength (dontmind / mild / strong), active pairs only")
s_right = (str_pred == str_true)
print(f"  strength correct:                       {pct(s_right[active].mean())}%")
print(f"  slot AND strength both correct:         {pct((s_right & (slot_pred == slot_true))[active].mean())}%   <- end to end")
for c, name in zip([1, 2, 3], ["dontmind", "mild", "strong"]):
    rows = active & (str_true == c)
    if rows.sum() > 0:
        print(f"    true {name:<9} {pct(s_right[rows].mean()):>5}%   (n = {int(rows.sum())})")
if active.sum() > 0:
    print(f"    overall       {pct(s_right[active].mean()):>5}%   (n = {int(active.sum())})")

In [ ]:
###################################################################
# Numbers: Accuracy, per-class precision / recall / F1, macro-F1
###################################################################
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

def report(title, y_true, y_pred, labels, names):
    print(f"\n=== {title} (n = {len(y_true)}) ===")
    print("confusion matrix (rows = true, cols = predicted):", names)
    # print(confusion_matrix(y_true, y_pred, labels=labels))
    print(classification_report(y_true, y_pred, labels=labels, target_names=names,
                                digits=3, zero_division=0))

# ---------------- direction: ALL cells, including gold-neutral ----------------
d_true = slot_true.ravel()                       # 310(N)*6 = 1,860 cells
d_pred = slot_pred.ravel()
report("Direction, all cells", d_true, d_pred, [0, 1, 2], ["negative", "neutral", "positive"])

stance_f1 = f1_score(d_true, d_pred, labels=[0, 2], average="macro", zero_division=0)
print(f"stance F1 (macro over negative/positive): {stance_f1:.3f}")
print(f"baseline, always 'neutral' accuracy:      {(d_true == 1).mean():.3f}")

# per RIASEC type: stance F1 on that type's N cells
print("\nper-type stance F1 (negative/positive, all cells of that type)")
for i in range(6):
    f = f1_score(slot_true[:, i], slot_pred[:, i], labels=[0, 2], average="macro", zero_division=0)
    print(f"  {TYPES[i]:<14} {f:.3f}")

# ---------------- strength: gold-active cells only ----------------
report("Strength, gold-active cells", str_true[active], str_pred[active],
       [1, 2, 3], ["dontmind", "mild", "strong"])
within1 = (abs(str_pred[active] - str_true[active]) <= 1).mean()
print(f"within one level: {within1:.3f}")

# ---------------- reason: one per sentence ----------------
report("Reason, all sentences", reason_true, reason_pred, [0, 1, 2],
       ["enjoyment", "familiarity", "neither"])